# MongoDB

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

MongoDB is a document database. Records are BSON documents stored in collections, the schema is flexible by default and enforceable with `$jsonSchema`, indexes make the common queries fast, replica sets provide automatic failover, and sharding provides horizontal scale.

## What is MongoDB?

MongoDB is a document database built on a horizontal scale-out architecture that uses a flexible schema for storing data. Founded in 2007, MongoDB has a worldwide following in the developer community.

The unit of storage is a BSON document: key/value pairs with nested objects, arrays, and rich types (dates, decimals, ObjectId, binary). Documents with a similar shape live in a collection. There is no fixed table schema, joins are optional (`$lookup`, `$graphLookup`), and scale-out is built in through replica sets and shards rather than bolted on.

```text
// mongosh: switch database, insert, query, project
use shop
db.products.insertOne({
  sku: "A1",
  name: "Espresso Machine",
  price: NumberDecimal("249.00"),
  tags: ["kitchen", "coffee"],
  stock: { warehouse: "DFW", qty: 12 }
})
db.products.find({ tags: "coffee" }, { name: 1, price: 1, _id: 0 }).sort({ price: -1 }).limit(5)
```

Core capabilities an interviewer expects you to name: document model, secondary indexes (single, compound, multikey, text, TTL, geospatial, wildcard, partial), the aggregation framework, multi-document transactions (4.0+), replica sets, sharding, and tunable read/write concern.

The mental model is a B-tree over documents rather than a table of rows. A collection is a set of BSON documents; each document is one contiguous byte string with a length prefix, so the server can skip over a document it does not need. A field can hold a scalar, a nested object, or an array, and an index on a field inside an array becomes a multikey index with one index entry per element.

BSON is a binary encoding, not text. Keys are repeated in every document, so a long field name costs bytes on every row and shows up in the working-set size. Types survive the round trip: `Decimal128` stays exact for money, `Date` stores UTC milliseconds, and `ObjectId` carries a timestamp plus a random counter.


```javascript
// BSON type choices matter; this document is ~120 bytes before compression
db.products.insertOne({
  sku: "A1",
  price: NumberDecimal("249.00"),   // exact, not a float64
  created_at: new Date(),
  tags: ["kitchen", "coffee"],      // multikey when indexed
  dimensions: { h: 32, w: 18 }      // embedded object
})
db.products.findOne({ sku: "A1" }, { _id: 0, sku: 1, price: 1 })
```


A common mistake is treating "flexible schema" as "no schema". Without a validator, one writer can store `price` as a string and another as `Decimal128`, and range queries then silently miss documents because BSON types are compared within a type ordering. Add `$jsonSchema` validation for the invariants you cannot live without, and decide field names deliberately because they are stored in every document.


## How do you model documents in MongoDB?

MongoDB modeling depends on access patterns. Embed related data when it is usually read together and has bounded size. Reference related data when it grows independently, is shared by many documents, or would create very large documents.

Example embedded order:

```json
{
  "_id": "order_1001",
  "customer_id": "C123",
  "items": [
    {"sku": "A1", "qty": 2},
    {"sku": "B9", "qty": 1}
  ]
}
```

This is efficient when the application usually reads an order with its items. For large or frequently changing child collections, referencing is safer.

The rule of thumb is one-to-few embeds, one-to-many references, and one-to-squillions (unbounded, such as events or messages) uses a separate collection plus the bucket pattern so a single document never grows without limit. References cost an extra round trip and an application-side join, or a `$lookup` in the pipeline.

```text
// Referenced model: order points at the customer
db.orders.insertOne({ _id: "order_1002", customer_id: "C123", total: NumberDecimal("312.00") })
```


Duplicate deliberately when reads dominate and the copied value changes rarely (for example the product name inside an order line). Duplication is a cache: you must decide who owns updates and how stale copies are refreshed.

The mechanics are a size and ownership decision, not a style preference. Embedding keeps one document as the aggregate and gives a single atomic read and write. Referencing splits the data into two collections joined by an application-side lookup or `$lookup`, which costs a round trip and cannot be updated atomically unless you use a transaction.

Bound the embed. The 16 MB document limit is hard, and an array that grows without limit also forces the storage engine to rewrite the document as it grows, which moves data and dirties more pages. The bucket pattern groups a fixed number of children per document, for example one document per device per hour, so documents stay a predictable size.


```javascript
// Bucket pattern: one document per sensor per hour, not one per reading
db.readings.insertOne({
  sensor_id: "s-42",
  bucket_start: ISODate("2026-10-04T13:00:00Z"),
  count: 360,
  readings: [ { ts: ISODate("2026-10-04T13:00:01Z"), temp: 21.4 } ]
})

// $lookup is the server-side join; it needs an index on the foreign field
db.orders.aggregate([
  { $match: { _id: "order_1001" } },
  { $lookup: { from: "customers", localField: "customer_id",
               foreignField: "_id", as: "customer" } }
])
```


The mistake to name in an interview is embedding a "one-to-squillions" relationship, such as every event for a user, because the array grows until it hits 16 MB and write latency degrades long before that. Also avoid duplicating a field that changes often: duplication is a cache, so every copy needs an owner and a refresh path, or the two copies drift.


## How do indexes affect MongoDB query performance?

Indexes let MongoDB find documents without scanning the whole collection. Compound indexes should match common filters and sort patterns. Too many indexes slow writes and consume storage, so indexes should be tied to real query patterns.

Example:

```text
db.orders.createIndex({ customer_id: 1, created_at: -1 })
```

This helps queries like "latest orders for customer C123." Use explain plans to confirm the index is used.

```text
const plan = db.orders
  .find({ customer_id: "C123" })
  .sort({ created_at: -1 })
  .explain("executionStats")
```


Two numbers matter in an interview: `nReturned` vs `nTotalDocsExamined` (a large ratio means the index is not selective enough) and `totalKeysExamined`. A `COLLSCAN` or an in-memory `SORT` stage in the winning plan is the signal to add or reorder an index. Index types worth naming: compound, multikey (an indexed array), text, TTL (auto-expiry), 2dsphere, partial (`partialFilterExpression`), unique, and wildcard.

Mechanically a MongoDB index is a B-tree keyed by the indexed fields plus the record id, stored as its own WiredTiger file. An `IXSCAN` walks that tree to a range of keys, then a `FETCH` stage reads the matching documents. A `COLLSCAN` reads every document in the collection. The planner picks the winning plan by racing candidate plans and keeping the one with the best score.

A compound index is usable when the query has an equality prefix on the leading fields. The order of the fields is the order of the sort inside the B-tree, so `{ customer_id: 1, created_at: -1 }` supports "one customer, newest first" but not a sort by `created_at` alone. The sort direction can be reversed by walking the tree backwards, but mixed directions inside one index only support that exact direction or its full reverse.


```javascript
const stats = db.orders.find({ customer_id: "C123" })
                       .sort({ created_at: -1 })
                       .explain("executionStats").executionStats
print(stats.nReturned)           // documents returned
print(stats.totalKeysExamined)   // index keys read
print(stats.totalDocsExamined)   // documents fetched
// totalKeysExamined must be >= nReturned; nReturned must equal totalDocsExamined
```


The common mistake is reading only `executionTimeMillis`, which varies with cache state, instead of the work counters. Another is adding an index for every query: each index must be updated on every insert, update, and delete, so a collection with twelve indexes performs thirteen writes per insert. Build the index for the query pattern you actually run, then confirm with `explain`.


## Why use MongoDB?

As a document database, MongoDB makes it easy for developers to store structured or unstructured data. It uses a JSON-like format to store documents. This format directly maps to native objects in most modern programming languages, making it a natural choice for developers, as they don't need to think about normalizing data. MongoDB can also handle high volume and can scale both vertically or horizontally to accommodate large data loads.

The practical drivers are: one round trip returns a whole aggregate (an order with its lines) instead of a multi-table join; the schema can evolve without a migration window; read scaling and failover are built into a replica set; and sharding extends the same model to multi-terabyte collections.

The engine-level reason is the aggregate: a document is the unit of atomicity and of locality, so a read that touches one document is one page fetch, while the same read in a normalized schema crosses several tables and several pages. A replica set then adds read capacity and automatic failover, and sharding adds write capacity and dataset size. Each property comes from a different component, so be precise about which one you are claiming.


The trade is that you must design around the query patterns you know. A document model optimized for one access path makes a different one awkward, and there is no cost-based join planner to rescue an unplanned `$lookup`. Choose MongoDB when the top queries are aggregate-shaped and the schema moves; choose a relational warehouse when the top queries are ad-hoc joins over many entities.


## What are the Advantages of MongoDB?

- A Powerful Document-Oriented Database: rich query language, aggregation framework, geospatial and text search, and multi-document transactions.
- Developer User Experience: documents map to objects in application code; no object-relational impedance mismatch during early product iteration.
- Scalability and Transactionality: replica sets for availability, sharding for horizontal scale, and snapshot-isolation transactions across documents and collections.
- Platform and Ecosystem Maturity: official drivers for every major language, Atlas managed service, Compass, BI connectors, and a large operational toolset.
- Flexible Schema: fields can differ per document, which fits evolving and heterogeneous data; validation can be added later where the data needs it.

The capabilities map to concrete server features worth naming: secondary indexes of many types, the aggregation pipeline for server-side transforms, multi-document transactions from 4.0 (replica sets) and 4.2 (sharded clusters), replica sets for failover, sharding for scale, and tunable read and write concern. The advantage is not that any single feature is unique; it is that the document model, the indexes, and the distribution model all agree on the same unit.


## When Should You Use MongoDB?

- Integrating large amounts of diverse data (catalogs with heterogeneous attributes, semi-structured ingestion).
- Describing complex data structures that evolve quickly and would otherwise force constant DDL migrations.
- Delivering data in high-performance applications where one document read replaces several joins.
- Supporting hybrid and multi-cloud applications with replica sets in multiple regions.
- Supporting agile development and collaboration, where a schema change is a code change.

When not to reach for it: heavy multi-entity transactional reporting, applications that need mature SQL analytics and star-schema joins, data that exceeds the 16 MB document limit as a single aggregate, or workloads with thousands of secondary indexes where write amplification dominates.

A quick test: write the three most frequent queries as documents. If most of them touch a single collection and return a document-shaped result, MongoDB fits. If they join four entities and aggregate across them, a relational engine or warehouse fits. Mixed workloads are normal, so many teams keep MongoDB as the operational store and stream changes into a warehouse for analytics.


The hard limits to check before committing are the 16 MB document size, the absence of foreign keys, and the fact that a unique index is per collection, so cross-collection uniqueness needs its own mechanism. None of these are blockers, but each one changes a design decision you would otherwise make implicitly.


## How does MongoDB exactly store the data?

In MongoDB, records are stored as documents in compressed [BSON](https://www.mongodb.com/json-and-bson) files. The documents can be retrieved directly in JSON format, which has many benefits:

- It is a natural form to store data.
- It is human-readable.
- Structured and unstructured information can be stored in the same document.
- You can nest JSON to store complex data objects.
- JSON has a flexible and dynamic schema, so adding fields or leaving a field out is not a problem.
- Documents map to objects in most popular programming languages.

Most developers find it easy to work with JSON because it is a simple and powerful way to describe and store data.

Under the hood the default storage engine is WiredTiger: documents are stored in compressed files (snappy by default for collections, zlib/zstd configurable), each collection has a `_id` index, and the journal (write-ahead log) provides durability. BSON supports types JSON does not, such as `ObjectId`, `Decimal128`, `Date`, `Binary`, and 64-bit integers, which matters for money and timestamps.

```text
// BSON size matters: one document may not exceed 16 MB
const doc = { sku: "A1", tags: ["kitchen", "coffee"], created: new Date() }
print(Object.bsonSize(doc))   // bytes, including field names
```

Two consequences interviewers look for: big arrays make documents grow on update (the document may move on disk, and the Storage engine must rewrite it), and the 16 MB cap is a hard design constraint that pushes unbounded data into a child collection.

WiredTiger is the storage engine. Its cache holds uncompressed pages for read and write, and it uses multi-version concurrency control, so readers never block writers. The default cache size is the larger of half of (RAM minus 1 GB) and 256 MB, which is why a server with 64 GB of RAM gets about 31.5 GB of cache and the rest is left for the operating system, connections, and the aggregation pipeline. The cache evicts clean pages easily and dirty pages only after they are written; when the dirty fraction crosses about 20 percent, application threads are recruited to help evict and latency rises.

Durability comes from two mechanisms that are often confused. The journal is a write-ahead log on disk; with `j: true` the write is acknowledged only after it reaches the journal, and the journal is flushed about every 100 ms by default. A checkpoint is a consistent snapshot of the data files, taken about every 60 seconds or after 2 GB of journal, and recovery replays the journal from the last checkpoint.


```javascript
// Cache counters: read these before blaming a query
const wt = db.serverStatus().wiredTiger.cache
print(wt["bytes currently in the cache"])
print(wt["maximum bytes configured"])
print(wt["pages read into cache"])
print(wt["tracked dirty bytes in the cache"])

// Checkpoint and transaction counters
db.serverStatus().wiredTiger.transaction["transaction checkpoint most recent time (msecs)"]

// Override the cache size at startup
// mongod --wiredTigerCacheSizeGB 8
```


The common mistake is setting the WiredTiger cache to almost all of RAM. That starves the file system cache, which WiredTiger also relies on, and the server starts swapping. The other mistake is assuming that `w: 1` means durable: `w: 1` without `j: true` can be lost in a power failure if the journal has not flushed, and only `j: true` or `w: "majority"` gets you a durability guarantee.


## MongoDB vs. RDBMS: What are the differences?

One of the main differences between MongoDB and RDBMS is that RDBMS is a relational database while MongoDB is nonrelational. Likewise, while most RDBMS systems use SQL to manage stored data, MongoDB uses BSON for data storage.

While RDBMS uses tables and rows, MongoDB uses documents and collections. In RDBMS a table -- the equivalent to a MongoDB collection -- stores data as columns and rows. Likewise, a row in RDBMS is the equivalent of a MongoDB document but stores data as structured data items in a table. A column denotes sets of data values, which is the equivalent to a field in MongoDB.

| Aspect | RDBMS | MongoDB |
|---|---|---|
| Unit | row in a table | BSON document in a collection |
| Schema | fixed, DDL migrations | flexible, optional `$jsonSchema` validation |
| Relationships | foreign keys + joins | embedding, `$lookup`, application joins |
| Transactions | multi-row by default | single-document atomic; multi-document with sessions |
| Scale-out | mostly vertical, read replicas | native sharding + replica sets |
| Query language | SQL | MQL / aggregation pipeline, SQL via Atlas SQL |
| Best fit | strongly relational, reporting | aggregate-oriented, evolving, high write volume |

MongoDB is not "RDBMS without discipline": the schema still exists, it just lives in the application and in validators rather than in foreign-key constraints.

The mechanical difference is where the work happens. A relational engine plans a join across tables using statistics and executes it in one statement. MongoDB reads one document, so "joining" is either embedded at write time or done by the application or by `$lookup`, which is not a cost-based join optimizer. That is a deliberate trade: you move work from read time to write time by denormalizing.

Transactions narrow the gap. A relational database gives multi-statement ACID by default; MongoDB gives atomicity per document by default and multi-document transactions when you opt in with a session, at the cost of a round trip and a 60-second default lifetime. For most document-shaped workloads, the single-document atomic guarantee is enough.


The mistake is to compare feature lists instead of access patterns. Ask what the top query looks like and how it changes: if it is a point read of an aggregate, MongoDB wins on locality; if it is an unplanned join across many entities, the relational optimizer wins. Schema discipline does not disappear either: the schema moves into validators, indexes, and application code where the relational database would have enforced it.


## How does mongodb scale?

mongo scales the same way as any distributed application would scale, throw more resources at it (`Vertical Scaling`) or distribute the data on multiple physical servers or nodes (`Horizontal Scaling`).

or in other words:

*Vertical Scaling* involves increasing the capacity of a single server, such as using a more powerful CPU, adding more RAM, or increasing the amount of storage space. It is simple but has a ceiling and is often a single point of failure.

*Horizontal Scaling* involves dividing the system dataset and load over multiple servers, adding additional servers to increase capacity as required. Sharding is MongoDB's horizontal mechanism; replica sets scale reads and provide availability but do not split a single collection.

While the overall speed or capacity of a single machine may not be high, each machine handles a subset of the overall workload, potentially providing better efficiency than a single high-speed high-capacity server.

Before sharding, exhaust the cheaper levers: correct indexes, covered queries, read preference `secondaryPreferred` for analytics, projection to avoid fetching large fields, and `$out`/`$merge` rollups so dashboards read small summary collections.

Vertical scaling is one bigger machine and has a hard ceiling plus a single failure domain. Horizontal scaling is more machines, and MongoDB's horizontal story has two layers that are easy to conflate. A replica set copies the entire dataset to each member, so it scales reads and availability only. A sharded cluster splits each collection into chunks on a shard key, so it scales writes and dataset size. If a single collection is too large or too hot, only sharding helps.


Before distributing, shrink the problem. The cheapest levers in order are: fix the index so the query stops scanning, project only the fields you need, move cold data out with a TTL or Atlas online archive, and precompute rollups with `$merge` so dashboards read a small collection. Sharding adds a router hop, a shard key you cannot easily change, and operational surface, so it should be the last lever, not the first.


## How does mongodb scale horizontally?

MongoDB supports *horizontal scaling* through [sharding.](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-sharding)

Sharding is a method for distributing data across multiple machines. MongoDB uses sharding to support deployments with very large data sets and high throughput operations.

A MongoDB [sharded cluster](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-sharded-cluster) consists of the following components:

- [shard](https://www.mongodb.com/docs/manual/core/sharded-cluster-shards/#std-label-shards-concepts): Each shard contains a subset of the sharded data. Each shard must be deployed as a [replica set.](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-replica-set)
- [mongos](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/): The `mongos` acts as a query router, providing an interface between client applications and the sharded cluster. [`mongos`](https://www.mongodb.com/docs/manual/reference/program/mongos/#mongodb-binary-bin.mongos) can support [hedged reads](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/#std-label-mongos-hedged-reads) to minimize latencies.
- [config servers](https://www.mongodb.com/docs/manual/core/sharded-cluster-config-servers/#std-label-sharding-config-server): Config servers store metadata and configuration settings for the cluster. As of MongoDB 3.4, config servers must be deployed as a replica set (CSRS).

The following graphic describes the interaction of components within a sharded cluster:

![Diagram of a sample sharded cluster for production purposes.  Contains exactly 3 config servers, 2 or more ``mongos`` query routers, and at least 2 shards. The shards are replica sets.](https://www.mongodb.com/docs/manual/images/sharded-cluster-production-architecture.bakedsvg.svg)

MongoDB shards data at the [collection](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-collection) level, distributing the collection data across the shards in the cluster.

```text
// Enable sharding and pick a shard key (run against mongos)
sh.enableSharding("shop")
sh.shardCollection("shop.orders", { customer_id: "hashed" })
```


A sharded cluster needs a shard key chosen up front: queries that include the shard key (or its prefix) are routed to one shard; everything else becomes a scatter-gather across all shards.

Mechanically, a sharded collection is divided into chunks, each a contiguous range of shard key values with a default maximum size of 128 MB. The `mongos` routers keep a cached copy of the chunk-to-shard map from the config servers, and the balancer moves chunks between shards when counts diverge. A query is targeted when it includes the shard key or a prefix of a compound shard key; otherwise `mongos` broadcasts to every shard and merges results, which multiplies latency by the slowest shard.

Shard keys cannot be changed on a collection before MongoDB 5.0; 5.0 adds `reshardCollection`, and 4.4 adds refining a shard key by adding a suffix field. This is why the key is a design decision made at creation time.


```javascript
sh.enableSharding("shop")
sh.shardCollection("shop.orders", { tenant_id: 1, order_id: 1 })

sh.status()                 // chunk distribution per collection
sh.getBalancerState()       // is the balancer running?
db.chunks.find({ ns: "shop.orders" }).count()   // chunk count
```


```json
{"queryPlanner": {"winningPlan": {"stage": "SHARD_MERGE", "shards": [
  {"shardName": "shardA", "winningPlan": {"stage": "IXSCAN"}},
  {"shardName": "shardB", "winningPlan": {"stage": "COLLSCAN"}}
]}}}
```


The common mistake is choosing a monotonically increasing shard key such as a timestamp or an `ObjectId`. Every new document lands in the highest chunk, so one shard takes all writes while the others idle, and the balancer cannot fix it because the hot range cannot be split into useful pieces. High cardinality with even write distribution matters more than raw cardinality alone.


## What are the advantages of sharding?

##### Reads / Writes

MongoDB distributes the read and write workload across the [shards](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-shard) in the [sharded cluster](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-sharded-cluster), allowing each shard to process a subset of cluster operations. Both read and write workloads can be scaled horizontally across the cluster by adding more shards.

For queries that include the shard key or the prefix of a [compound](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-compound-index) shard key, [`mongos`](https://www.mongodb.com/docs/manual/reference/program/mongos/#mongodb-binary-bin.mongos) can target the query at a specific shard or set of shards. These [targeted operations](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/#std-label-sharding-mongos-targeted) are generally more efficient than [broadcasting](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/#std-label-sharding-mongos-broadcast) to every shard in the cluster.

[`mongos`](https://www.mongodb.com/docs/manual/reference/program/mongos/#mongodb-binary-bin.mongos) can support [hedged reads](https://www.mongodb.com/docs/manual/core/sharded-cluster-query-router/#std-label-mongos-hedged-reads) to minimize latencies.

##### Storage Capacity

[Sharding](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-sharding) distributes data across the [shards](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-shard) in the cluster, allowing each shard to contain a subset of the total cluster data. As the data set grows, additional shards increase the storage capacity of the cluster.

##### High Availability

The deployment of config servers and shards as replica sets provide increased availability.

Even if one or more shard replica sets become completely unavailable, the sharded cluster can continue to perform partial reads and writes. That is, while data on the unavailable shard(s) cannot be accessed, reads or writes directed at the available shards can still succeed.

The cost of these advantages is operational: more components, a shard key that is expensive or impossible to change later, and the risk of an unbalanced cluster if the key is monotonically increasing (for example an ObjectId or a timestamp) or too coarse.

The availability advantage deserves precision. A shard is itself a replica set, so losing a secondary costs nothing, and losing a primary triggers an election inside that shard. If a whole shard replica set is down, reads and writes routed to the remaining shards still succeed, but any targeted write to the missing chunk fails, and a scatter-gather query that touches that range may return incomplete results unless you filter it out with `shardVersion` awareness. Config servers run as a replica set too, because losing the metadata makes the cluster unusable.


The cost is paid on every query that does not carry the shard key. Scatter-gather sends the query to every shard, so latency becomes the slowest shard and throughput is bounded by the slowest shard, not by total capacity. Target operations by including the shard key prefix in the query and in the leading fields of an index on each shard.


## What methods can we use for sharding in mongodb?

MongoDB supports two sharding strategies for distributing data across [sharded clusters.](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-sharded-cluster)

##### Hashed Sharding

Hashed Sharding involves computing a hash of the shard key field's value. Each [chunk](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-chunk) is then assigned a range based on the hashed shard key values. It spreads writes well even for monotonically increasing keys, but range queries on the shard key cannot be targeted and must be broadcast.

##### Ranged Sharding

Ranged sharding involves dividing data into ranges based on the shard key values. Each [chunk](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-chunk) is then assigned a range based on the shard key values. Range queries are targeted, but a monotonically increasing key sends every new write to the last chunk, creating one hot shard.

```text
// Hashed: even write spread, scatter-gather on ranges
sh.shardCollection("shop.events", { tenant_id: "hashed" })
```


A good shard key is high cardinality, low frequency per value, non-monotonic, and present in the queries you care about. Consider a compound key such as `{ tenant_id: 1, created_at: 1 }` when you need multi-tenant isolation plus recent-data locality.

Hashed sharding hashes the shard key value and stores chunks over the hash space, which spreads monotonically increasing keys evenly but destroys range locality. Ranged sharding keeps the natural order, so a range query on the shard key is targeted, but a monotonic key concentrates writes. A compound key is often the right compromise: a high-cardinality routing field first for even spread, then a date second for locality, for example `{ tenant_id: 1, created_at: 1 }`.

Zones let you pin ranges of the shard key to specific shards, which is how you keep European data in European data centers while still sharding globally. Since MongoDB 4.4 you can refine a shard key by appending a suffix field without resharding; since 5.0 `reshardCollection` can change the key entirely, but it is an online, expensive operation.


```javascript
// Hashed: even writes, but range queries broadcast
sh.shardCollection("shop.events", { device_id: "hashed" })

// Ranged compound: targeted by tenant, local by date
sh.shardCollection("shop.orders", { tenant_id: 1, created_at: 1 })

// Zone: keep EU tenants on EU shards
sh.addShardToZone("shardEU", "EU")
sh.updateZoneKeyRange("shop.orders", { tenant_id: "eu-a" }, { tenant_id: "eu-z" }, "EU")

// Refine a shard key by appending a field (4.4+)
db.adminCommand({ refineCollectionShardKey: "shop.orders",
                  key: { tenant_id: 1, created_at: 1, order_id: 1 } })
```


The mistake is choosing a shard key from the data model instead of from the query and write patterns. A key must appear in the queries you care about, or every query becomes scatter-gather, and it must have high cardinality with even frequency, or chunks cannot split and one shard takes the load. Low-cardinality keys such as `status` or `country` create chunks that can never split below the number of distinct values.


## How does atomicity and transaction word in mongodb?

In MongoDB, a write operation is [atomic](https://www.mongodb.com/docs/manual/reference/glossary/#std-term-atomic-operation) on the level of a single document, even if the operation modifies multiple embedded documents *within* a single document.

When a single write operation (e.g. [`db.collection.updateMany()`](https://www.mongodb.com/docs/manual/reference/method/db.collection.updateMany/#mongodb-method-db.collection.updateMany)) modifies multiple documents, the modification of each document is atomic, but the operation as a whole is not atomic.

For situations that require atomicity of reads and writes to multiple documents (in a single or multiple collections), MongoDB supports distributed transactions, including transactions on replica sets and sharded clusters. Transactions require a replica set or sharded cluster, use snapshot isolation, and default to a 60-second lifetime.

In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/?replicaSet=rs0")
accounts = client.bank.accounts
transfers = client.bank.transfers

with client.start_session() as session:
    with session.start_transaction():
        accounts.update_one({"_id": "A"}, {"$inc": {"balance": -100}}, session=session)
        accounts.update_one({"_id": "B"}, {"$inc": {"balance": 100}}, session=session)
        transfers.insert_one(
            {"from": "A", "to": "B", "amount": 100, "status": "posted"},
            session=session,
        )

Because a single-document update is atomic, many "transactions" are avoidable: `$inc`, `$push`, `$addToSet`, `findAndModify`, and upserts each operate atomically on one document. Design the document so the invariant fits inside it, and reach for a multi-document transaction only when it genuinely spans documents.

A single-document write is atomic because it is one BSON document mutated under a WiredTiger transaction; either the whole update applies or none of it does. Multi-document transactions use the same storage-engine transaction machinery extended across documents and shards with snapshot isolation: the transaction reads a consistent snapshot at a point in time and writes are invisible until commit. They require a replica set or sharded cluster, default to a 60-second lifetime, and are limited by a 16 MB total oplog entry for the transaction.


```javascript
// Atomic on one document: no transaction needed
db.accounts.updateOne(
  { _id: "acct-1", balance: { $gte: 100 } },     // guard in the filter
  { $inc: { balance: -100 }, $push: { ledger: { amount: -100, at: new Date() } } }
)

// Multi-document: explicit session and transaction
const session = db.getMongo().startSession()
session.startTransaction({ readConcern: { level: "snapshot" }, writeConcern: { w: "majority" } })
try {
  session.getDatabase("bank").accounts.updateOne({ _id: "A" }, { $inc: { balance: -100 } })
  session.getDatabase("bank").accounts.updateOne({ _id: "B" }, { $inc: { balance: 100 } })
  session.commitTransaction()
} catch (e) {
  session.abortTransaction()
  throw e
}
```


The common mistake is reaching for a transaction when a guarded single-document update would do. Putting the condition in the filter, for example `{ _id: "acct-1", balance: { $gte: 100 } }`, makes the check and the decrement atomic in one operation. The other mistake is a long transaction: it holds a snapshot and pins history, so under write load it can generate write conflicts and grow the oplog. Keep transactions short and retry on `TransientTransactionError`.


## What is an Aggregation Pipeline in mongodb?

An aggregation pipeline consists of one or more [stages](https://www.mongodb.com/docs/manual/reference/operator/aggregation-pipeline/#std-label-aggregation-pipeline-operator-reference) that process documents:

- Each stage performs an operation on the input documents. For example, a stage can filter documents, group documents, and calculate values.
- The documents that are output from a stage are passed to the next stage.
- An aggregation pipeline can return results for groups of documents. For example, return the total, average, maximum, and minimum values.

```text
db.orders.aggregate([
  { $match: { status: "paid", created_at: { $gte: ISODate("2026-10-01") } } },
  { $unwind: "$items" },
  { $group: {
      _id: { day: { $dateTrunc: { date: "$created_at", unit: "day" } }, sku: "$items.sku" },
      revenue: { $sum: { $multiply: ["$items.qty", "$items.price"] } },
      orders: { $sum: 1 },
      avg_qty: { $avg: "$items.qty" }
  } },
  { $sort: { "_id.day": 1, revenue: -1 } },
  { $merge: { into: "daily_sku_revenue", whenMatched: "replace", whenNotMatched: "insert" } }
], { allowDiskUse: true })
```

Put `$match` and `$limit` first so indexes can be used and fewer documents flow through the pipeline. Stage names to know: `$match`, `$project`, `$group`, `$sort`, `$unwind`, `$lookup`, `$facet` (several reports in one pass), `$bucket`, `$setWindowFields` (running totals), and `$merge`/`$out` for materialized views. `$lookup` does not use an index on the foreign collection's join field unless one exists, so index `foreignField`.

Stages come in two shapes. Streaming stages process one document at a time and pass it on: `$match`, `$project`, `$addFields`, `$unset`, `$limit`, `$skip`, and `$unwind`. Blocking stages must consume all input before they emit anything: `$group`, `$sort`, `$bucket`, `$facet`, and `$count`. A blocking stage means the pipeline cannot return a first result until the whole input is seen, and it holds memory or spills to disk.

Ordering is therefore a performance decision. Push `$match` and `$project` immediately after the source so fewer documents and fewer fields reach the blocking stages. A `$sort` placed before a `$match` sorts documents that will be thrown away. A `$match` before `$sort` lets the planner use an index for the sort and avoid the blocking SORT stage entirely.


```javascript
db.orders.aggregate([
  { $match: { tenant_id: "t1", status: "paid", created_at: { $gte: ISODate("2026-10-01") } } },
  { $project: { _id: 0, customer_id: 1, items: 1, created_at: 1 } },
  { $unwind: "$items" },
  { $group: { _id: "$items.sku", revenue: { $sum: { $multiply: ["$items.qty", "$items.price"] } },
              orders: { $sum: 1 } } },
  { $sort: { revenue: -1 } },
  { $limit: 20 }
], { allowDiskUse: true })
```


`$lookup` is the expensive stage. In the common `localField`/`foreignField` form the server probes the foreign collection once per input document, and if `foreignField` is not indexed that probe is a collection scan, turning an N-document pipeline into N scans. Always index the foreign field. The `let`/`pipeline` form runs a sub-pipeline per input document and is more flexible but not cheaper.

`$facet` runs several sub-pipelines over the same input in one pass, which is efficient for building several report sections at once, but each sub-pipeline shares the 100 MB memory limit and cannot contain `$out` or `$merge`. `allowDiskUse: true` lets blocking stages spill to disk past the memory limit, which prevents failure at the cost of disk I/O.


The common mistake is putting `$project` after `$group` and expecting a speedup; by then the documents are already aggregated. Another is using `$lookup` to emulate a join on every request when the same reference data could have been embedded or cached. A third is forgetting that `$sort` at the end of a large pipeline blocks and can spill, so a top-k question should usually end with `$sort` plus `$limit` rather than sorting everything and slicing in the application.


In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
# pymongo (review only)
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/?replicaSet=rs0")
orders = client.shop.orders

pipeline = [
    {"$match": {"tenant_id": "t1", "status": "paid"}},
    {"$project": {"_id": 0, "items": 1, "created_at": 1}},
    {"$unwind": "$items"},
    {"$group": {"_id": "$items.sku",
                "revenue": {"$sum": {"$multiply": ["$items.qty", "$items.price"]}}}},
    {"$sort": {"revenue": -1}},
    {"$limit": 10},
]

for row in orders.aggregate(pipeline, allowDiskUse=True):
    print(row)

explain = orders.database.command(
    "explain",
    {"aggregate": "orders", "pipeline": pipeline, "cursor": {}},
    verbosity="executionStats",
)
print(explain["stages"][0]["$cursor"]["executionStats"]["totalKeysExamined"])


## Where should I use an index in mongodb?

If your application is repeatedly running queries on the same fields, you can create an index on those fields to improve performance.

Although indexes improve query performance, adding an index has negative performance impact for write operations. For collections with a high write-to-read ratio, indexes are expensive because each insert must also update any indexes.

Concretely: index the fields used in equality filters, in sorts, and in the join key of `$lookup`; add a TTL index when data expires; use a text index for search; and use a partial index when only a small slice of documents is queried.

```text
db.sessions.createIndex({ expires_at: 1 }, { expireAfterSeconds: 0 })   // TTL
db.orders.createIndex(
  { tenant_id: 1, status: 1 },                                          // partial
  { partialFilterExpression: { status: "open" } }
)
db.products.createIndex({ name: "text", description: "text" }, { weights: { name: 10 } })
```

Index types, and the one-line reason each exists. A single-field index supports equality and sort on one field. A compound index supports a query on several fields and the ESR ordering. A multikey index is a compound or single index whose field holds an array; it stores one key per element, cannot cover a query, and cannot index two array fields in the same compound index. A text index supports `$text` search with stemming and weights, and a collection may have only one. A TTL index deletes documents after a date, runs a background monitor about every 60 seconds, and must be a single-field index on a date or an array of dates. A partial index indexes only documents matching `partialFilterExpression`, which makes it smaller and cheaper. A wildcard index (`$**`) indexes arbitrary or unknown field names, which is flexible but larger and slower than a targeted index. A unique index enforces uniqueness, and a sparse index (legacy) skips documents missing the field.


```javascript
db.users.createIndex({ email: 1 }, { unique: true })
db.orders.createIndex({ customer_id: 1, created_at: -1 })          // compound
db.products.createIndex({ tags: 1 })                                // multikey (tags is an array)
db.products.createIndex({ name: "text", description: "text" },
                        { weights: { name: 10, description: 2 } })
db.sessions.createIndex({ expires_at: 1 }, { expireAfterSeconds: 0 })  // TTL
db.orders.createIndex({ tenant_id: 1 }, { partialFilterExpression: { status: "open" } })
db.logs.createIndex({ "$**": 1 })                                   // wildcard
db.stores.createIndex({ location: "2dsphere" })                     // geospatial
db.events.createIndex({ ts: 1 }, { expireAfterSeconds: 7776000 })   // delete after 90 days
```


In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
# pymongo (review only: no server needed to read this)
from pymongo import MongoClient, ASCENDING, DESCENDING

client = MongoClient("mongodb://localhost:27017/?replicaSet=rs0")
orders = client.shop.orders

orders.create_index([("customer_id", ASCENDING), ("created_at", DESCENDING)],
                    name="customer_recent")
orders.create_index("expires_at", expireAfterSeconds=0)
orders.create_index([("tenant_id", ASCENDING)],
                    partialFilterExpression={"status": "open"})

for spec in orders.list_indexes():
    print(spec["name"], spec["key"])


The common mistake is adding a TTL index on a field that is not a date, or expecting a compound TTL index to work: TTL must be a single-field index on a date. Another is a partial index whose filter does not appear in the query, in which case the planner cannot use it. A third is believing a text index accelerates a regex query: `$regex` needs a normal index with an anchored pattern, while `$text` needs a text index and matches words, not substrings.


## How would you choose a indexing strategy in mongo and what are some common considerations we need to care about?

The best indexes for your application must take a number of factors into account, including the kinds of queries you expect, the ratio of reads to writes, and the amount of free memory on your system.

The best overall strategy for designing indexes is to profile a variety of index configurations with data sets similar to the ones you'll be running in production to see which configurations perform best. Inspect the current indexes created for your collections to ensure they are supporting your current and planned queries. If an index is no longer used, drop the index.

These are some considerations you need to take for your indexing strategy:

- [Use the ESR (Equality, Sort, Range) Rule](https://www.mongodb.com/docs/manual/tutorial/equality-sort-range-rule/#std-label-esr-indexing-rule)

  The ESR (Equality, Sort, Range) Rule is a guide to creating indexes that support your queries efficiently.

- [Create Indexes to Support Your Queries](https://www.mongodb.com/docs/manual/tutorial/create-indexes-to-support-queries/#std-label-create-indexes-to-support-queries)

  An index supports a query when the index contains all the fields scanned by the query. Creating indexes that support queries results in greatly increased query performance.

- [Use Indexes to Sort Query Results](https://www.mongodb.com/docs/manual/tutorial/sort-results-with-indexes/#std-label-sorting-with-indexes)

  To support efficient queries, use the strategies here when you specify the sequential order and sort order of index fields.

- [Ensure Indexes Fit in RAM](https://www.mongodb.com/docs/manual/tutorial/ensure-indexes-fit-ram/#std-label-indexes-ensure-indexes-fit-ram)

  When your index fits in RAM, the system can avoid reading the index from disk and you get the fastest processing.

- [Create Indexes to Ensure Query Selectivity](https://www.mongodb.com/docs/manual/tutorial/create-queries-that-ensure-selectivity/#std-label-index-selectivity)

  Selectivity is the ability of a query to narrow results using the index. Selectivity allows MongoDB to use the index for a larger portion of the work associated with fulfilling the query.

ESR in one example: the query `{ tenant_id: "t1", status: "open" }` sorted by `created_at` and filtered by `total: { $gt: 100 }` wants the index `{ tenant_id: 1, status: 1, created_at: -1, total: 1 }` - equality fields first in any order, then the sort field in its sort direction, then the range field.

```text
// Which indexes exist, and are they actually used?
db.orders.getIndexes()
db.orders.aggregate([{ $indexStats: {} }], { comment: "find unused indexes" })
// Look at "accesses.ops" == 0 to find candidates for dropping
```

The ESR rule is the sorting rule for compound index fields. Equality fields go first, in any order, because each one pins an exact key prefix. The sort field goes next, in the direction the query sorts, because the B-tree then delivers rows already ordered and the blocking SORT stage disappears. Range fields go last, because a range scans many keys and any field after it cannot be used for equality or ordering efficiently.

A covered query is one where the index alone answers the query: every field in the filter and in the projection is in the index, and `_id` is excluded unless it is part of the index. The plan then shows no FETCH stage and `totalDocsExamined` is 0, which removes the document read entirely. A multikey index cannot cover a query on the array field.


```javascript
// Query: status and tenant are equality, created_at is the sort, total is a range
db.orders.createIndex({ tenant_id: 1, status: 1, created_at: -1, total: 1 })

// Covered query: index contains everything, so no document fetch
db.orders.find(
  { tenant_id: "t1", status: "open" },
  { _id: 0, tenant_id: 1, status: 1, created_at: 1 }
).sort({ created_at: -1 }).explain("executionStats")

// Selectivity: compare keys examined with documents returned
const s = db.orders.find({ tenant_id: "t1" }).explain("executionStats").executionStats
print(s.totalKeysExamined, s.nReturned, s.totalKeysExamined / s.nReturned)
```


The mistake is ordering by "most selective field first" instead of by role. Selectivity matters among the equality fields, but a range field placed before the sort forces an in-memory sort and loses the index order. Another mistake is expecting `hint()` to fix a bad index: hinting an index that cannot satisfy the sort only moves the SORT stage around. Confirm with `executionStats` that `totalDocsExamined` equals `nReturned` and that the winning plan has no `SORT` stage.


## How does MongoDB ensure data consistency and reliability?

MongoDB ensures data consistency and reliability through several mechanisms:

- Write Concern: Allows you to specify the level of acknowledgment required from MongoDB for write operations.
- Replica Sets: Provides data redundancy and high availability.
- Journaling: Ensures data durability by writing operations to a journal before applying them to the data files.
- Data Validation: Allows you to enforce document structure.

The combination to reason about is write concern plus read concern plus read preference:

```text
// Durable, majority-acknowledged, journaled write
db.orders.insertOne(
  { _id: "order_1003", total: NumberDecimal("99.00") },
  { writeConcern: { w: "majority", j: true, wtimeout: 5000 } }
)
```


`w: 1` acknowledges from the primary only and is fast but can be rolled back after failover. `w: "majority"` survives failover. `readConcern: "majority"` returns data that will not be rolled back; `"snapshot"` gives a consistent point-in-time view and is what transactions use. `readPreference: "secondaryPreferred"` scales reads but can serve stale data (eventual consistency).

Write concern has two independent parts: how many members must acknowledge, and whether the acknowledgement waits for the journal. `w: 0` is fire-and-forget. `w: 1` acknowledges from the primary's memory. `w: "majority"` acknowledges once a majority of voting members have applied the write, so it survives a failover. Adding `j: true` makes the acknowledgement wait for the on-disk journal. `wtimeout` bounds the wait so a slow secondary does not block forever.

Read concern decides what data a read is allowed to see. `local` returns whatever the node has, which can be rolled back. `majority` returns data acknowledged by a majority and therefore not rolled back. `snapshot` returns a consistent point-in-time view and is what transactions use. `linearizable` reads the latest majority-acknowledged write and waits; it is the strictest and the slowest. Read preference chooses which member serves the read: `primary` is consistent, `secondaryPreferred` trades freshness for read scale.


A useful rule of thumb: use `w: "majority"` and `readConcern: "majority"` for anything you cannot reconstruct, and `w: 1` with secondary reads for caches, metrics, and feeds where a rollback is tolerable. Change streams and transactions both rely on the oplog, so their retention depends on oplog size.


The common mistake is assuming `w: "majority"` also means durable to disk; without `j: true` the journal may be up to about 100 ms behind, so a power loss can still lose acknowledged writes. The other mistake is reading from secondaries with default `local` read concern and then reasoning about the result as if it were current. Secondary reads are eventually consistent, and replication lag is a real, monitorable number.


In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
# pymongo (review only): pick concerns per operation
from pymongo import MongoClient
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern
from pymongo.read_preferences import ReadPreference

client = MongoClient("mongodb://localhost:27017/?replicaSet=rs0")
orders = client.shop.orders

# Durable write: acknowledged by a majority and journaled
orders.with_options(
    write_concern=WriteConcern(w="majority", j=True, wtimeout=5000)
).insert_one({"_id": "order_2001", "total": 99})

# Read that will not observe rolled-back data
current = orders.with_options(
    read_concern=ReadConcern("majority"),
    read_preference=ReadPreference.PRIMARY,
).find_one({"_id": "order_2001"})
print(current)

# Scale reads to secondaries, accepting eventual consistency
for doc in orders.with_options(
    read_preference=ReadPreference.SECONDARY_PREFERRED
).find({"status": "paid"}).limit(5):
    print(doc["_id"])


## What are MongoDB's backup and restore options?

MongoDB offers several backup and restore options:

- mongodump and mongorestore: Command-line tools for creating binary exports of database contents and restoring them.
- Filesystem snapshots: For creating point-in-time snapshots of data (only consistent if taken with the journal, or on a stopped/fsync-locked node).
- MongoDB Atlas Backup: Continuous backups and point-in-time recovery for cloud-hosted databases.
- MongoDB Ops Manager: For on-premises deployments, offering backup automation and point-in-time recovery.

```bash
# Logical backup of one database
mongodump --uri="mongodb://localhost:27017" --db=shop --out=/backup/2026-10-04

# Point-in-time-consistent backup of a replica set (captures oplog for replay)
mongodump --uri="mongodb://localhost:27017/?replicaSet=rs0" --oplog --out=/backup/rs-2026-10-04

# Restore into a different database name
mongorestore --uri="mongodb://localhost:27017" --nsFrom="shop.*" --nsTo="shop_restore.*" /backup/2026-10-04
```

mongodump is logical and slows down on multi-terabyte clusters; for large clusters prefer snapshots or Atlas continuous backup plus oplog replay, and always test a restore. For a single collection, `mongoexport`/`mongoimport` produce JSON/CSV but lose BSON types such as `Decimal128`.

The options differ by whether they capture a consistent point in time. A logical dump (`mongodump`) walks collections and can drift unless it runs with `--oplog`, which records oplog entries written during the dump so the restore can replay to a consistent point. A filesystem snapshot is only consistent if it includes the journal or the node was stopped or fsync-locked. Atlas continuous backup and Ops Manager capture the oplog continuously, which is what gives point-in-time recovery.


The common mistake is treating a backup as proven before restoring it. Test the restore into an isolated environment on a schedule, and check that indexes, validators, and users are recreated, since `mongodump` captures them but `mongoexport` does not. Also remember that `mongoexport` writes text and loses BSON types such as `Decimal128` and `Date` precision, so it is an interchange format, not a backup.


## How does MongoDB handle concurrency?

MongoDB handles concurrency through:

- Document-level locking: Allows multiple clients to read and write different documents simultaneously.
- WiredTiger storage engine: Provides document-level concurrency control and compression.
- Multi-version Concurrency Control (MVCC): Allows readers to see a consistent view of data without blocking writers.

Two keys on the application side:

```text
// Optimistic concurrency: version field guards against lost updates
const res = db.documents.updateOne(
  { _id: "doc1", version: 42 },
  { $set: { body: "new" }, $inc: { version: 1 } }
)
if (res.matchedCount === 0) {
  throw new Error("conflict: document changed since it was read")
}
```


An unindexed update that matches many documents still takes a collection-level intent and can cause write conflicts (`WriteConflict` errors) under high concurrency; keeping documents small and updates targeted by `_id` avoids most of it.

WiredTiger uses document-level concurrency with MVCC. Writers take a write conflict check on the document they mutate; readers see a snapshot and are never blocked. Two transactions writing the same document produce a `WriteConflict` that the driver retries. A slow operation that matches many documents still touches many documents, so long scans that write are the main source of conflicts.

The two application-side tools are optimistic concurrency with a version field, and idempotent retries. Never hold a transaction open across a user think-time or a network call to another service; keep the transaction to the database work and retry on transient errors.


The common mistake is an unindexed `updateMany` over a large collection. Each matched document is examined and rewritten, and the operation holds a collection-level intent that competes with other writers, so throughput collapses. Adding the filter to an index, and batching the update, keeps each transaction short. The other mistake is assuming a `WriteConflict` is fatal; it is retryable, and the driver's retryable writes already handle most cases.


## What security features does MongoDB offer?

Once again we have many options for security as well, like:

- Authentication: Supports various authentication mechanisms (SCRAM, x.509 certificates, LDAP, Kerberos).
- Authorization: Role-Based Access Control (RBAC) for fine-grained access control.
- Encryption: Supports encryption at rest and in transit (TLS/SSL).
- Auditing: Allows tracking of system and user activities.
- Network isolation: Supports IP whitelisting and VPC peering.

```text
// Least-privilege application user with a custom role
db.createRole({
  role: "ordersApp",
  privileges: [
    { resource: { db: "shop", collection: "orders" }, actions: ["find", "insert", "update"] }
  ],
  roles: []
})
db.createUser({
  user: "orders-svc",
  pwd: passwordPrompt(),
  roles: [{ role: "ordersApp", db: "shop" }]
})
```


Also name Client-Side Field Level Encryption (CSFLE) for sensitive fields such as PII or card numbers, and note that encrypting fields makes them unqueryable by equality unless you use deterministic encryption.

Security is layered and each layer is enforced by a different component. Authentication proves identity (SCRAM is the default, x.509 and LDAP for enterprise). Authorization is role-based: a role is a named set of privileges of the form `{ resource, actions }`, and users receive roles. Encryption in transit is TLS between clients and between replica set members; encryption at rest is file-level by default and can be client-side field level (CSFLE) for specific fields. Auditing records administrative and authentication events. Network controls are the outermost layer: bind to private interfaces, allow-list addresses, and require TLS between members.

CSFLE has a queryability consequence. Deterministic encryption produces the same ciphertext for the same plaintext, so equality queries still work; randomized encryption is stronger but the field becomes unqueryable, which means you must store a separate lookup token if you need to search it.


The common mistake is giving the application the built-in `readWrite` role on the whole database. Grant a custom role scoped to the collections and actions the service actually uses, and keep administrative users for humans with strong authentication. Store secrets outside the code and rotate the database credentials; the connection string is a credential.


## How does MongoDB manage schema design and changes?

MongoDB uses a flexible, document-based model for schema design:

- Schemaless: Collections don't enforce document structure by default.
- Dynamic Schema: Documents in a collection can have different fields.
- Schema Validation: Optional rules can be set to enforce document structure.
- Indexing: Supports various index types to optimize query performance.

For schema changes:

- Adding fields: Simply update documents with new fields.
- Removing fields: Delete fields from documents or use $unset in updates.
- Changing field types: Update documents with new data types.
- Large-scale changes: Can be done programmatically or using tools like Mongoose for Node.js.

```text
// JSON Schema validation: enforce shape, types, and required fields
db.createCollection("customers", {
  validator: {
    $jsonSchema: {
      bsonType: "object",
      required: ["email", "tier", "created_at"],
      additionalProperties: false,
      properties: {
        email: { bsonType: "string", pattern: "^.+@.+$" },
        tier: { enum: ["free", "pro", "enterprise"] },
        created_at: { bsonType: "date" },
        schema_version: { bsonType: "int", minimum: 1 }
      }
    }
  },
  validationLevel: "moderate",     // existing invalid docs are allowed
  validationAction: "error"        // or "warn"
})
```


Two patterns make changes safe. First, version documents (`schema_version`) so readers handle both shapes during a rollout. Second, expand-migrate-contract: add the new nullable field, dual-write, backfill with a batched update, then remove the old field. `validationLevel: "off"` or `"moderate"` avoids blocking writes mid-migration.

Schema change is a data migration, not a DDL statement, because there is no central catalog to alter. Adding a field means writing it, so you run a batched backfill that updates a bounded number of documents per round to avoid holding locks and to spread I/O. Removing a field is `$unset`, which can also be batched. Changing a type is the risky one: readers must handle both shapes while the backfill runs.

The safe sequence is expand, migrate, contract. Expand: add the new field and write both old and new shapes. Migrate: backfill in batches, verify counts, and only then switch readers. Contract: stop writing the old field and remove it later. Version each document with `schema_version` so code can branch during the window.


```javascript
// Batched backfill: bound each round by _id, never update the whole collection at once
let last = null, n = 0
while (true) {
  const filter = last ? { _id: { $gt: last }, schema_version: { $exists: false } }
                      : { schema_version: { $exists: false } }
  const docs = db.customers.find(filter).sort({ _id: 1 }).limit(1000).toArray()
  if (docs.length === 0) break
  db.customers.bulkWrite(docs.map(d => ({
    updateOne: { filter: { _id: d._id }, update: { $set: { schema_version: 2 } } }
  })), { ordered: false })
  last = docs[docs.length - 1]._id
  n += docs.length
}
print("migrated", n)
```


The common mistake is a single unbatched `updateMany` with no index on the filter. It scans the collection, rewrites every document in one operation, and can exhaust the oplog and block replication. Use a filter on an indexed field, a batch size in the low thousands, and `ordered: false` so one failure does not stop the rest. Run the validator with `validationLevel: "moderate"` during the migration so legacy documents do not block writes.


## What are the limitations or disadvantages of MongoDB?

Some limitations and disadvantages of MongoDB include:

- Limited JOIN functionality compared to relational databases (`$lookup` exists but is not a cost-based join optimizer).
- Document size limit of 16MB.
- Lack of built-in data integrity constraints (like foreign key constraints) - you enforce them with `$jsonSchema`, unique indexes, and transactions.
- Higher storage space requirements due to data duplication in denormalized models.
- Steeper learning curve for those accustomed to relational databases.
- Less mature ecosystem compared to some traditional RDBMSs, especially for BI, star-schema modeling, and ad-hoc analysts.
- Unbounded document growth: `$push` into an ever-growing array degrades updates and eventually hits 16 MB.
- No cross-collection unique constraint: a unique index is per collection, so global uniqueness needs an extra collection or a sharded check.
- Analytics: aggregations can be slow on huge collections without pre-aggregation, and there is no equivalent of a columnar warehouse.

Sort the limitations by what they cost you. The 16 MB document limit and unbounded array growth are design constraints. The absence of foreign keys and cross-collection uniqueness is an integrity constraint that you replace with validators, unique indexes, and reconciliation jobs. The lack of a cost-based join planner and columnar execution is an analytics constraint. Write amplification from many indexes is a throughput constraint.


The common mistake is discovering one of these at scale. Ask early: does this entity have an unbounded child list, does any invariant span two collections, and is the reporting workload ad-hoc joins? If yes to the last one, keep MongoDB as the operational store and stream changes into a warehouse with change streams rather than running dashboards on the operational cluster.


## How do you perform migrations from RDBMS to MongoDB?

Migrating from RDBMS to MongoDB typically involves these steps:

1. Analyze the relational schema and design a document model based on the top access patterns, not on the table diagram.
2. Map relational tables to MongoDB collections, deciding what to embed and what to reference.
3. Convert relational data to JSON or BSON, preserving types (decimals as `Decimal128`, timestamps as `Date`).
4. Use tools like mongoimport or write custom scripts to import data, and load in dependency order or in one pass for embedded aggregates.
5. Verify data integrity after migration: counts, checksums, and referential spot checks.
6. Add indexes and validators that mirror the constraints the RDBMS enforced (unique keys, not-null, enums).
7. Update application code to work with MongoDB instead of SQL.
8. Test thoroughly to ensure functionality and performance, then run both systems in parallel before cutover.

Tools that can assist in this process include:

- MongoDB Compass for visualizing and manipulating data.
- MongoDB Relational Migrator for schema mapping and CDC-based cutover.
- Official MongoDB connectors for various programming languages.
- Third-party ETL (Extract, Transform, Load) tools.

In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
# One-shot loader: read relational rows, reshape into documents, bulk insert
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
orders_col = client.shop.orders

# rows come from a SQL query that joins orders and order_items
docs = []
for order_id, customer_id, created_at, items in rows:
    docs.append({
        "_id": order_id,
        "customer_id": customer_id,
        "created_at": created_at,
        "items": [{"sku": s, "qty": q, "price": p} for s, q, p in items],
    })

orders_col.insert_many(docs, ordered=False)
orders_col.create_index([("customer_id", 1), ("created_at", -1)])

The risky part is not the copy but the cutover: keep relational IDs as `_id` so references remain stable, run dual writes or CDC, and decide how to handle transactions that spanned several tables.

Migration mechanics follow the access patterns. First list the top queries; then decide the aggregate boundaries. Usually one row becomes one document, but parent and child tables often merge into one embedded document, and many-to-many join tables become arrays of references or are embedded on one side. Preserve types: `NUMERIC` and `DECIMAL` become `Decimal128`, timestamps become `Date`, and relational ids stay as `_id` so references remain stable.

Cutover is the hard part. Either dual-write from the application, or capture changes with CDC and apply them to MongoDB, then verify by comparing counts, checksums, and spot checks on joined data. Keep the relational system as the source of truth until the comparison is clean, then switch, and keep the old system readable for a rollback window.


The common mistake is copying rows without preserving the constraints the source enforced. Unique keys, not-null checks, and enum domains must be recreated as unique indexes, `required` fields in `$jsonSchema`, and `enum` validators, or the new system will accept data the old one rejected. The other mistake is ignoring transactions that spanned tables in the source; those become multi-document transactions, or the invariant is redesigned into one document.


## What is MongoDB Atlas and what benefits does it provide?

MongoDB Atlas is the cloud-hosted database-as-a-service (DBaaS) platform for MongoDB. Benefits include:

- Automated deployment and scaling of MongoDB clusters.
- Multi-cloud and multi-region deployment options.
- Built-in security features (network isolation, encryption, access controls).
- Automated backups and point-in-time recovery.
- Performance monitoring and optimization tools, including the Performance Advisor and index suggestions.
- Easy integration with other cloud services.
- Reduced operational overhead for database management.
- Automatic updates and patches for the database software.

Atlas additions that come up in interviews: Atlas Search (Lucene-based full-text and vector search), Atlas Vector Search for RAG applications, Atlas Stream Processing, online archive to move cold data to object storage while keeping it queryable, and the Data API for HTTPS access without a driver.

Atlas is managed MongoDB: it provisions replica sets and sharded clusters, runs backups, exposes the Performance Advisor and index suggestions, and adds services that are not in the server, such as Atlas Search (Lucene), Atlas Vector Search, Atlas Stream Processing, and online archive that moves cold documents to object storage while keeping them queryable through the same connection.

The interview-relevant part is what Atlas does not change. The document model, indexes, shard keys, and read and write concerns behave the same, so a tuning problem on Atlas is still an `explain` problem. Atlas changes operations, not the query engine.


The common mistake is assuming a managed service removes the need to size anything. Atlas clusters still have a cache to fill and a working set to fit, and the Performance Advisor still flags a COLLSCAN. The other is using online archive as a substitute for a TTL: archived documents remain queryable and count toward storage, while a TTL deletes them.


## How can you monitor and optimize MongoDB performance?

Monitoring and optimizing MongoDB performance involves several strategies:

##### Monitoring:

- Use MongoDB's built-in tools like mongotop and mongostat
- Leverage MongoDB Compass for visual performance insights
- Implement MongoDB Atlas monitoring for cloud deployments
- Use third-party monitoring tools like Prometheus with MongoDB exporter

##### Optimization:

- Analyze and optimize slow queries using explain() method
- Create appropriate indexes based on query patterns
- Use proper schema design to minimize data duplication
- Configure appropriate write concern and read preferences
- Optimize server and storage configurations (WiredTiger cache, IOPS)
- Use appropriate shard keys for distributed systems

```text
// Find the slowest operations currently in flight
db.adminCommand({ currentOp: true, secs_running: { $gt: 1 } })
```


```bash
# Server-level counters and per-collection activity
mongostat --uri="mongodb://localhost:27017" --rowcount=5
mongotop 5
```

Look at disk IOPS and the WiredTiger cache hit ratio before blaming the query: a working set that does not fit in RAM will scan on disk even with a perfect index.

Monitoring has two layers. Server-level metrics tell you about resources: `db.serverStatus()` exposes connections, opcounters, and the WiredTiger cache; `mongostat` samples the counters once per interval; `mongotop` shows per-collection read and write time. Operation-level metrics tell you about queries: `db.currentOp()` shows what is running now, the slow query log records anything over `slowms`, and `$indexStats` shows which indexes are actually used.

Tuning follows the numbers, not a hunch. Read `explain("executionStats")` before and after a change, and compare the same counters. Watch the WiredTiger cache dirty fraction and page eviction rate: if the working set does not fit in cache, latency rises even when the plan reads the index correctly. Check replication lag and the oplog window, because a lagging secondary constrains `w: "majority"` latency.


The common mistake is tuning from `executionTimeMillis` alone, which changes with cache warmth and concurrency. Use `totalKeysExamined`, `totalDocsExamined`, and `nReturned`, which are deterministic for a dataset and plan. The other is profiling in production with level 2, which writes every operation into the system profile collection and becomes its own performance problem; use level 1 or a sampled profiler.


## What is a replica set in MongoDB and how does it work?

A replica set in MongoDB is a group of mongod processes that maintain the same data set. It provides:

- High Availability: If the primary node fails, an election occurs to choose a new primary from the secondary nodes.
- Data Redundancy: Data is replicated across multiple nodes.
- Read Scaling: Secondary nodes can handle read operations.

How it works:

- One primary node accepts all write operations
- Multiple secondary nodes replicate data from the primary via the oplog
- Optionally, arbiter nodes can participate in elections but don't hold data
- Automatic failover occurs if the primary becomes unavailable

```text
// Inspect and manage a replica set from mongosh
rs.status()
rs.conf()
rs.add("mongo3.example.com:27017")
rs.stepDown(60)                       // hand over the primary deliberately
```


Elections need a majority of voting members, so use an odd number of voters (3 or 5) and never two arbiters in a 3-node set. A replica set does not split data - every member holds the full collection - so it scales reads and availability, not write capacity or dataset size; sharding does that.

Replication works through the oplog, a capped collection in the `local` database. The primary records every write as an idempotent oplog entry; secondaries tail the oplog and apply entries in order, recording their position. A secondary's `optime` is how far it has applied, and replication lag is the difference between the primary's latest optime and a secondary's. Idempotency means an entry can be replayed safely after a restart, which is what makes the oplog the single mechanism behind replication, change streams, and retryable writes.

Failover is an election, not a promotion. Members exchange heartbeats; if a primary stops responding for `electionTimeoutMillis` (default 10 seconds), a member that is caught up and has a majority of votes calls an election. A majority of voting members must be reachable to elect a primary, which is why an even number of voters is worse than an odd number.


```javascript
rs.status().members.forEach(m => print(m.name, m.stateStr, m.optimeDate, m.health))
rs.printReplicationInfo()        // oplog size and time window
rs.printSecondaryReplicationInfo()

// Oplog window: how far behind a consumer may fall
db.getSiblingDB("local").oplog.rs.stats().maxSize
db.getSiblingDB("local").oplog.rs.find().sort({ $natural: -1 }).limit(1)

rs.stepDown(60)
```


Write concern and the journal interact with replication. `w: "majority"` waits for a majority of members to apply the write, which bounds failover data loss to writes below the commit point. `j: true` guarantees the primary wrote to its journal, which protects against a primary crash but not against losing the primary before replication. For the strongest combination, use `w: "majority"` with `j: true` and `readConcern: "majority"`.

The common mistake is running an even number of voting members or adding arbiters to break a tie. Two arbiters in a three-member set can elect a primary without any data-bearing member being current. Use three or five data-bearing voters, and use arbiters only to reach an odd voter count when a data-bearing member is not possible.


## How do change streams work in MongoDB?

Change streams expose a real-time, resumable stream of data changes (insert, update, replace, delete, and DDL events) without tailing the oplog manually. They require a replica set or a sharded cluster, and every event carries a `resumeToken` so a consumer resumes exactly where it stopped after a crash.

In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/?replicaSet=rs0")
orders = client.shop.orders

pipeline = [
    {"$match": {"operationType": {"$in": ["insert", "update"]}}},
    {"$project": {"fullDocument": 1, "operationType": 1, "ns": 1}},
]

resume_token = None
while True:
    with orders.watch(pipeline, resume_after=resume_token, full_document="updateLookup") as stream:
        for change in stream:
            resume_token = stream.resume_token
            publish_to_kafka(change["fullDocument"])   # or write to a search index

Use cases: syncing to Elasticsearch or a cache, emitting events to Kafka, invalidating application caches, and audit trails. `updateLookup` fetches the current version of the changed document. Oplog/change-stream retention bounds how long a consumer may be offline; if it falls behind the retention window the token is no longer valid and it must resynchronize.

Change streams are built on the oplog. The server opens a cursor over a filtered view of oplog entries, converts them to a stable event format, and returns a `resumeToken` with each event. Because the token is the oplog position, a consumer can resume exactly after the last processed event with `resumeAfter`, or after a timestamp with `startAtOperationTime`. Change streams require a replica set or sharded cluster, and their availability depends on the oplog retention window.

The event shape is stable across versions: `operationType` (`insert`, `update`, `replace`, `delete`, `invalidate`, and DDL types), `ns`, `documentKey`, `fullDocument` for insert and replace, and `updateDescription` for update. For an update you only get the changed fields unless you pass `fullDocument: "updateLookup"`, which fetches the current document. On a sharded cluster, each shard produces events and `mongos` merges them in cluster time order.


```javascript
const stream = db.orders.watch(
  [{ $match: { "fullDocument.status": "paid",
               operationType: { $in: ["insert", "update"] } } }],
  { fullDocument: "updateLookup", maxAwaitTimeMS: 1000 }
)

while (!stream.isClosed()) {
  if (stream.hasNext()) {
    const change = stream.next()
    print(change.operationType, change.documentKey._id, change.clusterTime)
    // change._id is the resume token for this event
  }
}
```


The common mistake is treating a change stream as durable queue storage. If the consumer stays offline longer than the oplog window, the resume token is no longer in the oplog and the stream fails with `ChangeStreamHistoryLost`; the consumer must resynchronize from a full read. Size the oplog for your worst acceptable downtime, monitor the oplog window, and persist the resume token only after the downstream write commits, so a crash reprocesses rather than skips. Make consumers idempotent for the same reason.


## How do you handle data integrity in MongoDB?

While MongoDB doesn't have built-in referential integrity like relational databases, you can maintain data integrity through:

- Schema Validation: Define JSON Schema to enforce document structure
- Application-Level Checks: Implement integrity checks in your application code
- Atomic Operations: Use multi-document transactions for operations that must succeed or fail as a unit
- Unique Indexes: Ensure uniqueness of certain fields
- Data Validation: Use $jsonSchema operator to define validation rules
- Consistent Data Modeling: Design schemas to minimize the need for complex integrity constraints

```text
// Uniqueness is an index, not an assumption
db.users.createIndex({ email: 1 }, { unique: true })
```


Referential integrity across collections is a trade-off, not a bug: you either pay for transactions on write, or you accept orphan risk and run a periodic reconciliation job. Referential cascade deletes must be implemented by the application or by a change-stream consumer.

Integrity in MongoDB is enforced per document by default. A unique index enforces uniqueness and doubles as the guard for idempotent inserts. `$jsonSchema` enforces shape, types, and required fields at write time. A single-document update keeps an invariant atomic. Anything spanning documents needs a transaction, and anything spanning collections needs either a transaction or an accepted orphan risk plus reconciliation.

Referential integrity is therefore a choice. Pay on write with a transaction, or pay later with a periodic job that finds and repairs orphans. A common pattern is to store a reference and a denormalized copy of the fields you read, then treat the copy as a cache with a defined refresh owner and an audit query that detects drift.


The common mistake is assuming a unique index on a field that can be missing works as expected. In MongoDB, a missing field is indexed as `null`, so a unique index rejects the second document that also lacks the field unless the index is partial or sparse. Use a partial index such as `{ partialFilterExpression: { email: { $exists: true } } }` when only present values must be unique.


## How does MongoDB fit into the CAP theorem?

The CAP theorem states that a distributed system can only provide two of three guarantees: Consistency, Availability, and Partition tolerance. MongoDB's position in the CAP theorem can be described as:

**Partition Tolerance**: MongoDB is designed to handle network partitions in distributed deployments.
**Consistency vs. Availability**: MongoDB allows you to tune the balance between consistency and availability:

With default settings, MongoDB leans towards **CP** (Consistency and Partition Tolerance)
By adjusting write concern and read preferences, you can shift towards **AP** (Availability and Partition Tolerance)

In practice:

- For strong consistency, use majority write concern and read preference
- For higher availability at the cost of potential inconsistency, use lower write concerns or read from secondaries

```text
// CP-leaning read after a majority write: never see rolled-back data
db.orders.insertOne({ _id: "order_1004", total: 10 }, { writeConcern: { w: "majority" } })
db.orders.find({ _id: "order_1004" }).readConcern("majority").readPref("primary")
```


Nuance worth stating: a partition that removes the majority of voters makes a replica set unavailable for writes (no primary can be elected) even though minority nodes still answer reads. That is a CP choice. Setting `w: 1` and reading secondaries shifts the same deployment toward availability at the cost of possible rollback visibility.

Map the guarantees to settings. A partition happens regardless of configuration, so the real choice is what the majority side does and what the minority side does. The majority side can elect a primary and continue accepting writes; the minority side cannot elect one, so its members become read-only. That is the CP behavior you get by default with `w: "majority"` and `readConcern: "majority"`. Choosing `w: 1` and reading from secondaries moves you toward AP: writes succeed on the majority side sooner, minority-side secondaries can still answer reads, but a failover can roll back recent writes and those reads can be stale.


The common mistake is calling MongoDB simply "CP". It is tunable per operation, and the deployment's behavior depends on the concerns you set, not on a fixed label. Say instead: with `w: "majority"` and `readConcern: "majority"` it behaves as CP; with `w: 1` and secondary reads it trades consistency for availability. Also note that losing the majority makes the set unavailable for writes no matter how you tune the client, because an election cannot complete.


## How do you handle large data volumes in MongoDB?

MongoDB offers several strategies for handling large data volumes:

- Sharding: Distribute data across multiple machines to handle large datasets and high throughput operations.
- Indexing: Create appropriate indexes to improve query performance on large collections.
- Aggregation Pipeline: Use for efficient data processing and analysis on large datasets, with `$merge` to maintain pre-aggregated rollups.
- GridFS: Store and retrieve large files (above 16 MB) by chunking them across two collections.
- Data Compression: WiredTiger storage engine provides data compression to reduce storage requirements.
- Capped Collections: Use for high-volume data that doesn't require long-term storage (legacy; time series collections are usually better).
- Time Series Collections: Optimize storage and querying for time series data.
- TTL indexes: Expire documents automatically so the hot set stays small.
- Online archive: Atlas moves cold documents to S3 while leaving them queryable.

```text
// Time series collection with automatic expiry
db.createCollection("sensor_readings", {
  timeseries: {
    timeField: "ts",
    metaField: "sensor",
    granularity: "minutes"
  },
  expireAfterSeconds: 7776000      // 90 days
})
```


The decision framework: reduce (TTL, rollups), then split (bucket pattern, time series collections), then distribute (sharding). Archiving to cheaper storage is often cheaper than growing the cluster.

Follow an order that stops as early as possible. Reduce first: TTL indexes delete expired documents, and `$merge` rollups replace raw events with small summaries. Split second: the bucket pattern groups many readings into one document, and time series collections store measurements in compressed buckets with a `metaField` for the series identity and an optional `expireAfterSeconds` for retention. Distribute third: shard on a key with high cardinality and even writes, ideally the same field that scopes the common query. Archive last: Atlas online archive moves cold documents to object storage while keeping them queryable.

Time series collections are the modern answer for measurements. They bucket by time automatically, compress well, and support the same query language, so the bucket pattern is usually only needed when you must update inside the bucket.


The common mistake is sharding first. Sharding adds a router hop and a shard key you cannot easily change, and it does not fix a workload that is slow because of a bad index or an unbounded array. The second mistake is deleting history in the application with a range delete, which leaves the storage unreclaimed until compaction; a TTL index and time series collections handle expiry without a delete job.


## What are the best practices for designing a MongoDB schema?

Key best practices for MongoDB schema design include:

- Embed related data in a single document when possible for faster reads.
- Use references when data is used in multiple places or for very large datasets.
- Design schema based on application query patterns, not on entity-relationship purity.
- Avoid deeply nested documents (keep nesting to 2-3 levels for better performance).
- Use appropriate data types for fields (`Decimal128` for money, `Date` for time, `ObjectId` for IDs).
- Create indexes to support common queries and verify them with `explain`.
- Consider document growth when designing schemas; never let an array grow unboundedly.
- Use schema versioning for easier updates and migrations.
- Normalize data only when necessary (e.g., for data consistency across multiple collections).
- Use array fields for one-to-many relationships within reasonable limits.
- Enforce the invariants you cannot live without with `$jsonSchema` and unique indexes.
- Store the field names you query on; long, deeply repeated field names cost bytes on every document.

The best practices reduce to three questions asked per collection. What is the aggregate that must be read or written atomically? How bounded is it? Which queries must be indexed, and in what order? Embed when the child is bounded and owned by the parent; reference when it is shared, unbounded, or large. Keep nesting shallow so dotted field names stay readable and updates stay targeted.

Then make the invariants explicit: `$jsonSchema` for shape and required fields, unique and partial indexes for uniqueness, and a version field for migrations. Choose types deliberately, `Decimal128` for money and `Date` for time, because comparability depends on it.


The common mistake is a design that silently grows. An array appended on every event, or a document that holds a count plus a list that could be derived, both become rewrites and eventually hit 16 MB. Cap arrays with `$slice`, move unbounded children into their own collection, and use the bucket pattern when a fixed window is acceptable.


## How does MongoDB handle geographic data and geospatial queries?

MongoDB provides robust support for geographic data and geospatial queries:

- Geospatial Indexes: Support for 2d and 2dsphere indexes for efficient geospatial queries.
- GeoJSON Objects: Store location data using standard GeoJSON format.
- Geospatial Queries: Support for various types of geospatial queries:
  - Proximity: Find documents near a point.
  - Intersection: Find geometries that intersect with a specified geometry.
  - Within: Find geometries contained within a specified geometry.
- Aggregation Pipeline: Geospatial stages like $geoNear for complex geo-based analytics.
- Coordinate Systems: Support for both legacy coordinate pairs and GeoJSON objects.
- Geospatial Operators: $near, $geoWithin, $geoIntersects, etc., for flexible querying.
- Spherical Geometry: 2dsphere index supports queries on an Earth-like sphere.

```text
db.stores.createIndex({ location: "2dsphere" })
```


GeoJSON coordinates are `[longitude, latitude]` - reversing them is the classic bug. A `2dsphere` index also supports `$geoWithin` (for example, stores inside a delivery polygon) and `$geoIntersects` (which zones a route crosses).

A `2dsphere` index indexes GeoJSON points and shapes on a sphere and supports `$near`, `$nearSphere`, `$geoWithin`, and `$geoIntersects`. A legacy `2d` index works on flat coordinate pairs and is rarely what you want. The index itself is a B-tree over a space-filling curve of the geometry, so a proximity query walks nearby cells. The `$geoNear` aggregation stage must be the first stage of the pipeline and requires a geospatial index.

Coordinates are always `[longitude, latitude]` in GeoJSON, in that order. A polygon must be a closed ring with the first and last positions equal, and a `2dsphere` query uses meters for radius in `$maxDistance` when the distance is in `$geoNear` with a spherical index.


The common mistake is swapping longitude and latitude, which puts the point in the wrong hemisphere and silently returns nothing. The other is using `$near` in an aggregation pipeline: `$near` is a query operator and is not allowed there, so use `$geoNear` as the first stage instead.


## What are the trade-offs and when should you not use MongoDB?

- Consistency vs availability: `w: "majority"` plus `readConcern: "majority"` is durable and rollback-safe, but each write waits for a majority round trip and the cluster refuses writes if it loses the majority. `w: 1` with secondary reads is faster and more available, but a failover can roll writes back and replicas can be stale.
- Write amplification: every secondary index must be updated on every write, and a multikey index is updated once per array element. A collection with 12 indexes turns one insert into 13 index writes; on write-heavy workloads, indexes are often the bottleneck, not the disk.
- Hot partitions and hot shards: a monotonically increasing shard key (timestamp, ObjectId) concentrates writes in the last chunk; a low-cardinality key (status, country) creates chunks that cannot split. Hashed sharding spreads writes but makes range queries scatter-gather.
- Document growth and the 16 MB limit: unbounded `$push` arrays cause rewrites and eventually hard failures. Cap arrays with `$slice`, or move children into their own collection.
- Memory pressure: index and working-set size must fit in RAM. Once the WiredTiger cache is exhausted, `IXSCAN` still reads from disk and latency jumps even though the plan looks correct.
- Weak analytics: no cost-based join planner, no columnar execution. If the workload is mostly arbitrary ad-hoc joins and aggregations across many entities, a warehouse or a relational database is the better home for it.
- When NOT to use MongoDB: strictly relational, constraint-heavy transactional systems (banking cores, double-entry ledgers needing foreign keys and multi-table ACID by default); heavy OLAP/BI with star-schema joins; workloads requiring cross-collection uniqueness or complex cascading deletes; teams without the operational maturity to run replica sets and sharding; and any case where a small relational database with a handful of tables would simply work.

The trade-offs are concrete settings and shapes, not slogans. Each one has a direction: majority write concern and majority read concern cost latency and availability for durability and rollback safety; every index costs write throughput for read speed; a monotonic shard key costs write distribution for range locality; denormalization costs storage and update discipline for read locality. Name the direction you chose and why.


The common mistake is choosing MongoDB for a workload with no aggregate. If every query joins five entities and aggregates across them, you have rebuilt a relational schema without the optimizer. The honest answer in an interview is the boundary: MongoDB for the operational aggregate, a warehouse for the joins and ad-hoc analytics, connected by change streams.


## How do you read explain("executionStats")?


Tuning starts by making the plan visible. `explain()` has three modes: `queryPlanner` (the winning plan and the rejected candidates, no execution), `executionStats` (runs the winning plan and reports work counters), and `allPlansExecution` (runs the candidates too, for comparing). Use `queryPlanner` to see which index won, and `executionStats` to see how much work the winner did. In a sharded cluster, run `explain` on `mongos` so you see the merge stage and each shard's plan.

Read three counters together. `nReturned` is how many documents the query returned. `totalKeysExamined` is how many index keys were read. `totalDocsExamined` is how many full documents were fetched. For a good query, `totalDocsExamined` equals `nReturned` and `totalKeysExamined` is close to `nReturned`. A large `totalKeysExamined / nReturned` ratio means the index is not selective enough for the filter; a large `totalDocsExamined / nReturned` ratio means the index found many candidates that the filter then rejected.


```javascript
const ex = db.orders.find({ customer_id: "C123", status: "paid" })
                    .sort({ created_at: -1 })
                    .explain("executionStats")

print(ex.queryPlanner.winningPlan.stage)
print(JSON.stringify(ex.queryPlanner.winningPlan, null, 2))
const s = ex.executionStats
print("nReturned", s.nReturned)
print("totalKeysExamined", s.totalKeysExamined)
print("totalDocsExamined", s.totalDocsExamined)
print("executionTimeMillis", s.executionTimeMillis)
print("returned per key", s.nReturned / Math.max(s.totalKeysExamined, 1))
```


```json
{
  "queryPlanner": {
    "winningPlan": {
      "stage": "FETCH",
      "inputStage": { "stage": "IXSCAN", "indexName": "customer_id_1_created_at_-1" }
    }
  },
  "executionStats": {
    "nReturned": 3,
    "executionTimeMillis": 0,
    "totalKeysExamined": 3,
    "totalDocsExamined": 3,
    "executionStages": {
      "stage": "FETCH",
      "nReturned": 3,
      "docsExamined": 3,
      "inputStage": { "stage": "IXSCAN", "keysExamined": 3, "nReturned": 3 }
    }
  }
}
```


The stage names to recognize: `COLLSCAN` means no index was used; `IXSCAN` is an index range scan; `FETCH` reads the documents the index pointed at; `SORT` is a blocking in-memory sort; `PROJECTION_COVERED` or an `IXSCAN` with no `FETCH` parent means the index covered the query; `SHARD_MERGE` and `SINGLE_SHARD` are the router stages. A `SORT` stage with `usedDisk: true` means the sort spilled, which `allowDiskUse` permits and `memLimit` limits.

A `COLLSCAN` is not automatically wrong: a query that returns most of a small collection is cheaper as a scan. The ratio decides. On a large collection, `totalDocsExamined` in the millions with a small `nReturned` is the signal to add or reorder an index.


The common mistake is comparing `executionTimeMillis` across runs and concluding an index helped. Timing depends on cache warmth and concurrency; the work counters are stable. Compare the same plan before and after a change and confirm that `totalKeysExamined` and `totalDocsExamined` moved in the right direction.


## How do you design indexes with the ESR rule and covered queries?


Write the query in terms of roles before writing the index. Equality predicates pin exact key prefixes. The sort field determines the order the B-tree must deliver. Range predicates scan a contiguous span. ESR orders them Equality, then Sort, then Range. Equality fields can be in any relative order, though putting the most selective first can help when the planner chooses a prefix. The sort field must follow in the requested direction so the blocking `SORT` stage disappears. A range field placed last still narrows the scan, but any field after a range cannot be used efficiently.

Covered queries are the other lever. If the index contains every field the query filters on and projects, the server answers from the index alone: no `FETCH` stage and `totalDocsExamined` is 0. Exclude `_id` in the projection unless it is part of the index. A multikey index cannot cover a query on the array field, and an index cannot cover a query that needs a document field that is not in the index.


```javascript
// Query roles: tenant_id and status are equality, created_at is the sort, total is a range
db.orders.createIndex({ tenant_id: 1, status: 1, created_at: -1, total: 1 },
                      { name: "esr_orders" })

// Covered: index contains filter and projection fields, _id excluded
db.orders.find(
  { tenant_id: "t1", status: "open" },
  { _id: 0, tenant_id: 1, status: 1, created_at: 1 }
).sort({ created_at: -1 }).explain("executionStats")

// Verify which index the planner chose, and force one when testing
db.orders.find({ tenant_id: "t1" }).hint("esr_orders").explain("executionStats")

// Find unused indexes before dropping anything
db.orders.aggregate([{ $indexStats: {} }])
```


The common mistake is ordering fields by perceived selectivity, for example putting a range date before an equality status because the date is "more selective". That forces an in-memory sort and may scan more keys. The other is expecting a covered query while projecting `_id`; `_id` is fetched unless the index includes it or the projection excludes it. A third is creating several near-duplicate compound indexes; each one adds write cost and the planner may pick the wrong one.


## How do you optimize an aggregation pipeline?


Think in terms of streaming and blocking stages. A streaming stage transforms each document as it arrives. A blocking stage must see all input before emitting, holds memory, and can spill. `$match`, `$project`, `$addFields`, `$unwind`, `$limit`, and `$skip` stream; `$group`, `$sort`, `$bucket`, `$facet`, and `$count` block. The optimizer can move a `$match` before a `$project` and can push a `$match` ahead of an `$unwind` when the predicate does not depend on the array, but it cannot rescue a pipeline whose stages are in the wrong order.

Put `$match` and `$project` first so indexes are used and fewer fields reach the blocking stages. Prefer `$sort` plus `$limit` for top-k, which the server can satisfy with an index and a bounded heap. Use `allowDiskUse: true` when a `$group` or `$sort` would otherwise exceed the 100 MB memory limit per stage, and watch for `usedDisk` in the explain output because spilling costs I/O.


```javascript
db.orders.aggregate([
  { $match: { tenant_id: "t1", created_at: { $gte: ISODate("2026-10-01") } } },
  { $project: { items: 1, customer_id: 1, created_at: 1, _id: 0 } },
  { $unwind: "$items" },
  { $group: { _id: "$items.sku",
              revenue: { $sum: { $multiply: ["$items.qty", "$items.price"] } } } },
  { $sort: { revenue: -1 } },
  { $limit: 10 },
  { $merge: { into: "sku_revenue", whenMatched: "replace", whenNotMatched: "insert" } }
], { allowDiskUse: true })
```


`$lookup` cost is the other lever. In the `localField`/`foreignField` form the server probes the foreign collection once per input document. With an index on `foreignField` each probe is a B-tree lookup; without one it is a collection scan, so an N-document pipeline does N scans. Index the foreign field, and put a `$match` before the `$lookup` so fewer documents trigger it. The `let`/`pipeline` form runs a sub-pipeline per input document and is not cheaper.

`$facet` runs several sub-pipelines over the same input in one pass, which avoids re-reading the source for several report sections. Each facet shares the memory limit, so heavy facets can spill or fail, and `$out` and `$merge` are not allowed inside a facet. Put the facet late, after filtering.


```javascript
// $lookup is cheap only when the foreign field is indexed
db.customers.createIndex({ _id: 1 })                       // already there
db.customers.createIndex({ region: 1 })                    // for the let/pipeline form

db.orders.aggregate([
  { $match: { created_at: { $gte: ISODate("2026-10-01") } } },
  { $lookup: { from: "customers", localField: "customer_id",
               foreignField: "_id", as: "customer" } },
  { $unwind: "$customer" },
  { $facet: {
      byRegion: [{ $group: { _id: "$customer.region", orders: { $sum: 1 } } }],
      topCustomers: [{ $group: { _id: "$customer_id", spend: { $sum: 1 } } },
                     { $sort: { spend: -1 } }, { $limit: 10 }]
  } }
], { allowDiskUse: true })
```


The common mistake is a `$lookup` with no index on `foreignField`, which turns one aggregation into a collection scan per input document. The next is `$project` after `$group`, which saves nothing. The third is `allowDiskUse: true` everywhere as a substitute for fixing the order: it prevents the failure but the disk spill is still slow, and the explain output shows `usedDisk`.


## How do you choose a shard key and size the WiredTiger cache?


A shard key is judged on cardinality, frequency, and monotonicity. Cardinality: enough distinct values that chunks can split and spread. Frequency: no single value dominates, or one chunk holds most documents and cannot split below one value per chunk. Monotonicity: values that increase over time send every insert to the highest chunk, so one shard takes all writes. The key must also appear in the queries you care about, or those queries become scatter-gather.

Hashed sharding fixes monotonicity by hashing the value, at the cost of range locality. Ranged compound keys keep locality: route on a high-cardinality field and include a date for range queries, for example `{ tenant_id: 1, created_at: 1 }`. Chunks are about 128 MB by default, and the balancer migrates chunks when counts diverge; zone rules pin ranges to shards for data residency.


```javascript
sh.shardCollection("shop.orders", { tenant_id: 1, created_at: 1 })
sh.status()
sh.getBalancerState()

// How many chunks per shard, to spot a hot shard
db.getSiblingDB("config").chunks.aggregate([
  { $group: { _id: "$shard", chunks: { $sum: 1 } } },
  { $sort: { chunks: -1 } }
])

// Balancer window: move chunks only during the quiet period
use config
db.settings.updateOne({ _id: "balancer" },
  { $set: { activeWindow: { start: "02:00", stop: "05:00" } } }, { upsert: true })
```


Size the WiredTiger cache before blaming the plan. The default is the larger of half of (RAM minus 1 GB) and 256 MB, so a 64 GB host gets about 31.5 GB. Set `wiredTigerCacheSizeGB` explicitly only when you have a reason, and leave headroom for the file system cache and connections. Watch `db.serverStatus().wiredTiger.cache` for the dirty fraction and the eviction rate: a working set larger than cache produces page evictions and disk reads even when the index is correct. Compress collections (snappy by default, zstd for better ratio) to fit more in cache and on disk.

Read and write concerns trade latency for safety on the same cluster. `w: "majority"` with `j: true` and `readConcern: "majority"` survives failover and never reads rolled-back data, at the cost of a majority round trip. `w: 1` with `readPreference: "secondaryPreferred"` is faster and more available but can serve stale reads and lose recent writes on failover.


The common mistake is picking the shard key last. It is set when the collection is sharded, and changing it needs an online reshard. The second is sizing cache to nearly all RAM, which starves the file system cache. The third is using `w: "majority"` everywhere without measuring the latency cost; put the strong concerns on the writes that must survive and keep caches and metrics on weaker ones.


## MongoDB exercises


These exercises use only the Python standard library and model the tuning arithmetic with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.


The numbers mirror the fields MongoDB reports. Index efficiency is `nReturned / totalKeysExamined`, a plan is diagnosed from the stage name and the work counters, WiredTiger's default cache is the larger of half of (RAM minus 1 GB) and 256 MB, and TTL retention is expressed in seconds. Treat the asserted values as checks on the formula, not as predictions of a real deployment.


### Try it 1. Order compound index fields by the ESR rule


**Task.** Given query fields tagged as equality, sort, or range, build the index key order that follows the ESR rule.


**Hint.** Equality fields first, then the sort field with its direction, then range fields.


**Expected result.** `[('status', 1), ('tenant_id', 1), ('created_at', -1), ('total', 1)]`.


In [ ]:
FIELDS = [
    {"name": "created_at", "role": "sort", "direction": -1},
    {"name": "status", "role": "equality"},
    {"name": "total", "role": "range"},
    {"name": "tenant_id", "role": "equality"},
]


def esr_index(fields):
    ordered = [f for f in fields if f["role"] == "equality"]
    ordered += [f for f in fields if f["role"] == "sort"]
    ordered += [f for f in fields if f["role"] == "range"]
    return [(f["name"], f.get("direction", 1)) for f in ordered]


key = esr_index(FIELDS)
print("index key:", key)

assert key == [("status", 1), ("tenant_id", 1), ("created_at", -1), ("total", 1)]
assert key[-1][0] == "total", "range fields must come last"


### Try it 2. Compute index selectivity from nReturned and totalKeysExamined


**Task.** Compute index efficiency as `nReturned / totalKeysExamined` for a poorly selective plan and for a good one, and classify each.


**Hint.** Efficiency near 1 means almost every key read produced a returned document; below about 0.5 means the index is examined too broadly.


**Expected result.** The poor plan has efficiency `0.00025` and verdict `reorder the index`; the good plan has efficiency `0.962` and verdict `index is selective`.


In [ ]:
def index_efficiency(n_returned, total_keys_examined):
    return n_returned / total_keys_examined


def verdict(efficiency):
    return "index is selective" if efficiency >= 0.5 else "reorder the index"


poor = index_efficiency(25, 100000)
good = index_efficiency(50, 52)
print(f"poor  plan: nReturned=25  totalKeysExamined=100000 -> {poor:.5f} ({verdict(poor)})")
print(f"good  plan: nReturned=50  totalKeysExamined=52     -> {good:.3f} ({verdict(good)})")

assert round(poor, 5) == 0.00025
assert round(good, 3) == 0.962
assert verdict(poor) == "reorder the index"
assert verdict(good) == "index is selective"


### Try it 3. Detect a COLLSCAN from an explain summary


**Task.** Given an execution summary with the winning stage and the work counters, classify the plan.


**Hint.** `COLLSCAN` or a zero `totalKeysExamined` means no index was used. Otherwise compare `totalDocsExamined` and `totalKeysExamined` with `nReturned`.


**Expected result.** The summary classifies as `COLLSCAN: add an index on the filter fields`.


In [ ]:
def diagnose(plan):
    if plan["stage"] == "COLLSCAN" or plan["totalKeysExamined"] == 0:
        return "COLLSCAN: add an index on the filter fields"
    if plan["totalDocsExamined"] > 10 * plan["nReturned"]:
        return "IXSCAN is not selective: reorder the index"
    if plan["totalKeysExamined"] > 10 * plan["nReturned"]:
        return "index too coarse: add an equality field"
    return "plan is healthy"


summary = {
    "stage": "COLLSCAN",
    "nReturned": 12,
    "totalKeysExamined": 0,
    "totalDocsExamined": 1240000,
}

result = diagnose(summary)
print("stage:", summary["stage"])
print("docs examined per returned:", summary["totalDocsExamined"] // summary["nReturned"])
print("verdict:", result)

assert result == "COLLSCAN: add an index on the filter fields"
assert diagnose({"stage": "IXSCAN", "nReturned": 50, "totalKeysExamined": 52,
                 "totalDocsExamined": 50}) == "plan is healthy"


### Try it 4. Choose a shard key from cardinality and write distribution


**Task.** Score candidate shard keys by cardinality and write spread, reject candidates that are not in the queries or have too few distinct values, and pick the best.


**Hint.** A usable key must appear in the queries and have at least a thousand distinct values; among usable keys, prefer high cardinality and even write spread, and penalize monotonic keys by their low write spread.


**Expected result.** `tenant_id` wins with a score near `2.97`; `status` is rejected.


In [ ]:
import math

CANDIDATES = [
    {"name": "created_at", "cardinality": 1_000_000, "write_spread": 0.10, "in_queries": True},
    {"name": "tenant_id", "cardinality": 2_000, "write_spread": 0.90, "in_queries": True},
    {"name": "status", "cardinality": 5, "write_spread": 0.95, "in_queries": False},
]


def score(candidate):
    if not candidate["in_queries"] or candidate["cardinality"] < 1000:
        return -1.0
    return math.log10(candidate["cardinality"]) * candidate["write_spread"]


ranked = sorted(CANDIDATES, key=score, reverse=True)
for c in ranked:
    print(f"{c['name']:12s} score={score(c):.3f}")
print("chosen shard key:", ranked[0]["name"])

assert ranked[0]["name"] == "tenant_id"
assert round(score(CANDIDATES[1]), 2) == 2.97
assert score(CANDIDATES[2]) == -1.0, "status is not in the queries"


### Try it 5. Estimate the WiredTiger cache from RAM


**Task.** Compute the default WiredTiger cache size for hosts with 64 GB, 8 GB, and 2 GB of RAM.


**Hint.** The default is the larger of half of (RAM minus 1 GB) and 256 MB; 256 MB is 0.25 GB.


**Expected result.** `31.5` GB, `3.5` GB, and `0.5` GB (the 256 MB floor for the 2 GB host after the subtraction).


In [ ]:
def wiredtiger_cache_gb(ram_gb):
    return max(0.5 * (ram_gb - 1), 0.25)


for ram in (64, 8, 2):
    cache = wiredtiger_cache_gb(ram)
    print(f"RAM {ram:>2} GB -> cache {cache:>5.2f} GB")

assert wiredtiger_cache_gb(64) == 31.5
assert wiredtiger_cache_gb(8) == 3.5
assert wiredtiger_cache_gb(2) == 0.5


### Try it 6. Size a TTL index from retention days


**Task.** Convert a 90-day retention policy into the `expireAfterSeconds` value and estimate how many readings the index must track for a fixed ingest rate.


**Hint.** Seconds per day is 86400. Documents retained is `per_second * 86400 * days`; the TTL monitor deletes on a background pass, so a document can outlive its expiry by up to about 60 seconds.


**Expected result.** `expireAfterSeconds` is `7776000`, about `31.1` billion readings are retained, and the monitor delay bound is `60` seconds.


In [ ]:
def ttl_seconds(days):
    return days * 86400


retention_days = 90
expire_after = ttl_seconds(retention_days)
readings_per_second = 4000
retained = readings_per_second * 86400 * retention_days
ttl_monitor_delay_seconds = 60

print("expireAfterSeconds:", expire_after)
print("retained readings: %.1f billion" % (retained / 1e9))
print("TTL monitor delay bound:", ttl_monitor_delay_seconds, "seconds")

assert expire_after == 7776000
assert round(retained / 1e9, 1) == 31.1
assert ttl_monitor_delay_seconds == 60


### Try it 7. Model the cost of $lookup from collection sizes


**Task.** Given an orders collection and a customers collection, compare the index probes an indexed `$lookup` performs with the document reads an unindexed one performs.


**Hint.** With an index on `foreignField` the server probes once per input document; without one it scans the foreign collection once per input document.


**Expected result.** The indexed form does about `2,000,000` reads (one probe plus one fetch per order) and the unindexed form `500,000,000,000`; the unindexed form is `250000` times more work.


In [ ]:
orders = 1_000_000
customers = 500_000

indexed_reads = orders * 2          # one index probe plus one foreign document fetch per order
unindexed_reads = orders * customers  # a full scan of customers per order
slowdown = unindexed_reads / indexed_reads

print("orders:", orders, "customers:", customers)
print("indexed lookups  :", indexed_reads)
print("unindexed lookups:", unindexed_reads)
print("slowdown factor  :", int(slowdown))

assert indexed_reads == 2_000_000
assert unindexed_reads == 500_000_000_000
assert slowdown == 250000.0


### Try it 8. Triage a plan by documents examined per returned document


**Task.** Compute `totalDocsExamined / nReturned` for an execution summary and decide whether the index is earning its keep.


**Hint.** A healthy index scan examines one document per returned document; a ratio far above 1 means the index found many candidates that the filter rejected.


**Expected result.** The ratio is `100.0` and the verdict is `poor selectivity: examine fewer documents per result`.


In [ ]:
def triage(n_returned, total_docs_examined):
    ratio = total_docs_examined / n_returned
    if ratio <= 1.5:
        return ratio, "healthy: about one document per result"
    return ratio, "poor selectivity: examine fewer documents per result"


ratio, verdict = triage(n_returned=12, total_docs_examined=1200)
print(f"docsExamined/nReturned = {ratio:.1f}")
print("verdict:", verdict)

assert ratio == 100.0
assert verdict == "poor selectivity: examine fewer documents per result"
assert triage(50, 50)[0] == 1.0


## Real world use cases for MongoDB

Each case below states the business context, the access patterns, the document model, and why that model fits. Read them as worked design decisions: the numbers drive the choice of embedding versus referencing, the index, and the shard key.


### 1. E-commerce product catalog

Business context: a marketplace with 8 million SKUs, each category having different attributes (clothes have sizes and colors, electronics have voltage and warranty). Volume: ~8M products, ~25M variants, read-heavy at roughly 50:1 reads to writes, 3,000 reads/s at peak. Access patterns: fetch one product page with all variants and images, search by category and tags, update price and stock frequently.

Schema design: one document per product embedding variants and a bounded image list; references to a `categories` collection so category metadata is not duplicated 8M times. A `2dsphere` index is not needed; a compound index on `{ category_id: 1, updated_at: -1 }` serves category browsing.

```text
db.products.insertOne({
  _id: "sku-1001",
  name: "Espresso Machine",
  category_id: "cat-kitchen",
  attributes: { voltage: "110V", warranty_months: 24 },
  tags: ["coffee", "kitchen"],
  variants: [
    { sku: "sku-1001-black", color: "black", price: NumberDecimal("249.00"), stock: 12 },
    { sku: "sku-1001-steel", color: "steel", price: NumberDecimal("269.00"), stock: 3 }
  ],
  updated_at: new Date()
})
```


Why this design: the product page is a single-document read, and `variants` is bounded (a few dozen at most), so embedding wins. The heterogeneous `attributes` object is exactly the flexible-schema case a relational catalog needs EAV tables for.

### 2. IoT time series with automatic expiry

Business context: 40,000 sensors reporting every 10 seconds (about 345M readings/day, tens of billions over the retention window). Access patterns: dashboards read one sensor for the last 24 hours, rarely query across all sensors; older than 90 days is discarded.

Schema design: a time series collection with `metaField` on the sensor and a TTL, or the bucket pattern (one document per sensor per hour) when you need sub-document updates.

```text
db.createCollection("sensor_readings", {
  timeseries: { timeField: "ts", metaField: "sensor", granularity: "seconds" },
  expireAfterSeconds: 7776000
})
```


Why this design: time series collections store measurements columnar-ish by bucket internally, compress well, and TTL removes expired data without a delete job. Sharding uses `{ sensor.id: "hashed" }` or a compound `{ "sensor.id": 1, ts: 1 }` so writes spread across shards while a single-sensor range query stays targeted.

### 3. User activity feed

Business context: a social app with 50M users; the timeline of a user with 5,000 followers must load in under 100 ms. Access patterns: read the newest 50 items for one user, append new items at high rate, keep only the last few hundred items.

Schema design: fan-out on write into a per-user feed document that keeps a bounded array using `$push` with `$slice`; if a user's feed can exceed a few thousand items, use a dedicated `feed_items` collection keyed by `{ user_id, created_at }`.

```text
// Bounded timeline inside one document
db.timelines.updateOne(
  { _id: "user-42" },
  {
    $push: {
      items: {
        $each: [{ actor: "user-7", verb: "liked", ts: new Date() }],
        $position: 0,
        $slice: 500
      }
    }
  },
  { upsert: true }
)
```


Why this design: reads dominate, so precomputing the feed on write (`$push` with `$slice`) removes the read-time fan-out entirely. For celebrities whose feeds would exceed 16 MB, the separate collection with a compound index keeps documents bounded and lets a shard key of `{ user_id: 1 }` localize each feed.

### 4. Fintech ledger and idempotent payments

Business context: a payments API writing double-entry ledger entries and transferring funds between accounts. Volume: ~5M transactions/day, strict correctness, auditable history. Access patterns: look up a transaction by idempotency key, list the last 100 entries for an account, move money between two accounts atomically.

Schema design: `accounts` and `ledger_entries` collections; a unique index on the idempotency key so a retried request cannot double-post; a multi-document transaction for the transfer; `w: "majority"` everywhere.

In [ ]:
# ILLUSTRATIVE - requires pymongo: install to run, or read on.
from pymongo import MongoClient, ReturnDocument

client = MongoClient("mongodb://localhost:27017/?replicaSet=rs0")
db = client.ledger
db.payments.create_index("idempotency_key", unique=True)

with client.start_session() as session:
    with session.start_transaction(
        read_concern={"level": "snapshot"},
        write_concern={"w": "majority"},
    ):
        db.payments.insert_one(
            {"idempotency_key": "req-7f3a", "status": "posted", "amount": 5000},
            session=session,
        )
        db.accounts.update_one(
            {"_id": "acct-1"}, {"$inc": {"balance": -5000}}, session=session
        )
        db.accounts.update_one(
            {"_id": "acct-2"}, {"$inc": {"balance": 5000}}, session=session
        )
        db.ledger_entries.insert_many(
            [
                {"account_id": "acct-1", "amount": -5000, "ref": "req-7f3a"},
                {"account_id": "acct-2", "amount": 5000, "ref": "req-7f3a"},
            ],
            session=session,
        )

Why this design: `$inc` is atomic and the unique idempotency index makes retries safe; the transaction spans collections only because the accounting invariant demands it. Replicas give durability, and change streams feed the downstream audit and reporting systems.

### 5. Multi-tenant SaaS

Business context: a B2B platform with 2,000 tenants and wildly different data volumes (one tenant is 30% of all rows). Volume: 900M documents, retention differs per tenant. Access patterns: every query is scoped to one tenant; schema varies slightly per tenant plan; older data expires after the contract ends.

Schema design: a shared collection with `tenant_id` in every document and in the leading position of every index; per-tenant `$jsonSchema` variants are avoided in favor of a superset validator plus optional fields; TTL or archival per tenant.

```text
// Every index leads with tenant_id so reads are always targeted
db.tickets.createIndex({ tenant_id: 1, status: 1, created_at: -1 })
```


Why this design: a shared collection with a tenant prefix is the cheapest multi-tenant model and makes isolation a matter of discipline plus a mandatory index prefix. Truly regulated or enormous tenants can be moved to their own database and cluster without changing the application's tenant-scoped access pattern.